# Stage C 03l — C19 Google Cloud handoff

This companion leaves notebook 03l unchanged. Stop its training process before export. The cutover lock forbids concurrent Colab training; the final cell only publishes a fully verified, completed E25 run. The VM stops after 36 hours or successful completion. Budget alerts do not stop spending; the SSD and Cloud Storage remain billable and must be reviewed for manual cleanup after return. Cleanup is never automatic: use `gcloud compute instances delete stage-c-c19 --zone ZONE --keep-disks=data`, then (only after verified Drive return) `gcloud compute disks delete stage-c-c19 --zone ZONE` and `gcloud storage rm --recursive gs://BUCKET/stage-c-c19`.


In [ ]:
# REQUIRED OPERATOR INPUTS (the first executable cell)
GIT_REF='main'  # cloud-support branch; C19 training itself is pinned separately to ae72fae…
PROJECT_ID=input('Google Cloud project ID: ').strip()
BUCKET=input('Cloud Storage bucket name (without gs://): ').strip()
REGION=input('Preferred region [auto]: ').strip() or 'auto'
DRIVE_ROOT=input('Drive root [/content/drive/MyDrive/SeqTrainerStageC]: ').strip() or '/content/drive/MyDrive/SeqTrainerStageC'
if not PROJECT_ID or not BUCKET: raise ValueError('Project ID and bucket are required')


In [ ]:
from pathlib import Path
from google.colab import auth, drive
import json, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
auth.authenticate_user()
subprocess.run(['gcloud','config','set','project',PROJECT_ID],check=True)
repo=Path('/content/SeqTrainer-cloud-support')
if not repo.exists(): subprocess.run(['git','clone','https://github.com/Gonza10V/SeqTrainer.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',f'origin/{GIT_REF}'],check=True)
launcher=repo/'scripts/stage_c_c19_gcp.py'
if not launcher.is_file(): raise FileNotFoundError('Cloud-support launcher is not present on the selected support branch')
# Cloud systemd logs replace seqtrainer-titans-stage-c-colab-run for this unattended handoff.
print('Training must be stopped in 03l. Export will refuse LIVE_STATUS=running.')


In [ ]:
# Stable-copy only the declared dataset, panels, C18 result, study records, and full C19 run.
exports=Path('/content/c19_gcp_cutovers')
command=[sys.executable,str(launcher),'export','--drive-root',DRIVE_ROOT,'--output',str(exports),'--bucket',BUCKET]
subprocess.run(command,check=True)
bundles=sorted(exports.glob('step-*'))
if not bundles: raise RuntimeError('No cutover was produced')
BUNDLE=bundles[-1]
CUTOVER=json.loads((BUNDLE/'CUTOVER_MANIFEST.json').read_text())
print('Immutable cutover:',CUTOVER['cutover_id'])
print('Resume step:',CUTOVER['checkpoint']['optimizer_step'])
print('Post-checkpoint telemetry (archived, not resumable):',len(CUTOVER['post_checkpoint_telemetry']))

# Validate billing, both GPU quotas, bucket, checksums, disk sizing, and the $250 working cap.
preflight=[sys.executable,str(launcher),'preflight','--project',PROJECT_ID,'--bucket',BUCKET,'--region',REGION,'--bundle',str(BUNDLE)]
subprocess.run(preflight,check=True)
print('Preflight passed. Provisioning is deliberately a separate operator action.')
print('Run:',' '.join([sys.executable,str(launcher),'provision','--project',PROJECT_ID,'--bucket',BUCKET,'--region',REGION,'--bundle',str(BUNDLE)]))


In [ ]:
# FINAL RETURN — run only after status reports completion. Publication is atomic and guarded.
if 'BUNDLE' in globals() and (BUNDLE/'CUTOVER_MANIFEST.json').is_file():
    manifest=BUNDLE/'CUTOVER_MANIFEST.json'
else:
    lock=json.loads((Path(DRIVE_ROOT)/'runs/c19_v3_medium_adaptive_e25/GCP_CUTOVER_LOCK.json').read_text())
    manifest=Path('/content/CUTOVER_MANIFEST.json')
    subprocess.run(['gcloud','storage','cp',f"gs://{BUCKET}/stage-c-c19/cutovers/{lock['cutover_id']}/CUTOVER_MANIFEST.json",str(manifest)],check=True)
subprocess.run([sys.executable,str(launcher),'repatriate','--bucket',BUCKET,'--drive-root',DRIVE_ROOT,'--cutover-manifest',str(manifest)],check=True)
print('Verified completed C19 bundle published to Drive. The pre-cutover checkpoint is archived.')
